<a href="https://colab.research.google.com/github/vzuza/mapa_restec_ceped/blob/main/mapa_residentes_restec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [181]:
# INSTALAÇÃO, IMPORTS E CONFIGURAÇÕES

%pip -q install geopandas contextily matplotlib pandas requests unidecode folium plotly openpyxl

import re
import json
import zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests
import folium
import plotly.express as px
from google.colab import files
from pathlib import Path
from unidecode import unidecode
from branca.element import Element
from getpass import getpass

In [183]:



# ============================================================
# CAMINHOS
# ============================================================

ARQUIVO_XLSX = Path("/content/Tabela_Municipio_Area_Forca_Residentes.xlsx")

PASTA_SAIDA = Path("/content/mapas_restec")
PASTA_MALHA = Path("/content/malha_municipal_PR")

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
PASTA_MALHA.mkdir(parents=True, exist_ok=True)

# ============================================================
# MAPA-BASE: OPENSTREETMAP
# ============================================================

# Não necessita de API Key
MAPA_BASE = "OpenStreetMap"

# Não solicitar mais a chave do MapTiler
MAPTILER_KEY = ""

# ============================================================
# IDENTIDADE VISUAL
# ============================================================

AZUL = "#14566B"
AZUL_ESCURO = "#173D52"
AZUL_CLARO = "#A9C7DC"
VERMELHO = "#C8553D"
CINZA = "#AEBBC4"
FUNDO = "#F3F6F8"

PALETA_FORCAS = [
    "#14566B", "#C8553D", "#2A9D8F",
    "#D9962B", "#6A4C93", "#4D7EA8", "#7A8B3A"
]

# ============================================================
# MALHA MUNICIPAL IBGE 2022
# ============================================================

URL_MALHA = (
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/"
    "malhas_territoriais/malhas_municipais/"
    "municipio_2022/UFs/PR/PR_Municipios_2022.zip"
)


In [184]:

# LEITURA E LIMPEZA DA BASE RESTEC

def padronizar_nome(valor):
    if pd.isna(valor):
        return None

    return re.sub(
        r"\s+", " ",
        unidecode(str(valor)).upper().strip()
    )


def carregar_base(caminho):

    df = pd.read_excel(caminho, dtype="string")

    df.columns = df.columns.str.strip()

    df["Forca"] = df["Forca"].str.strip()
    df["area_atuacao"] = df["area_atuacao"].str.strip()

    df["residentes"] = pd.to_numeric(
        df["residentes"],
        errors="coerce"
    ).fillna(0)

    df["chave_municipio"] = df["MUN"].map(
        padronizar_nome
    )

    return df.dropna(
        subset=["chave_municipio", "Forca"]
    ).copy()


df = carregar_base(ARQUIVO_XLSX)

residentes_mun = (
    df.groupby("chave_municipio", as_index=False)
    ["residentes"].sum()
)

forcas_mun = (
    df.groupby(["chave_municipio", "Forca"], as_index=False)
    ["residentes"].sum()
)

print("VALIDAÇÃO DA BASE RESTEC")
print("-------------------------")
print("Registros:", len(df))
print("Residentes:", int(df["residentes"].sum()))
print("Municípios:", len(residentes_mun))
print("Forças:", df["Forca"].nunique())
print("Áreas de atuação:", df["area_atuacao"].nunique())

display(
    residentes_mun.nlargest(10, "residentes")
)


VALIDAÇÃO DA BASE RESTEC
-------------------------
Registros: 220
Residentes: 335
Municípios: 26
Forças: 7
Áreas de atuação: 31


,chave_municipio,residentes
8,CURITIBA,123
14,LONDRINA,28
15,MARINGA,25
4,CASCAVEL,23
20,PONTA GROSSA,22
9,FOZ DO IGUACU,20
11,GUARAPUAVA,17
10,FRANCISCO BELTRAO,13
24,UMUARAMA,13
7,CRUZEIRO DO OESTE,8


In [187]:
from google.colab import files
arquivos = files.upload()

Saving logo_restec.png to logo_restec.png
Saving logo_unespar.png to logo_unespar.png


In [188]:
import shutil

shutil.move(
    "/content/logo_unespar.png",
    "/content/mapas_restec/logo_unespar.png"
)

shutil.move(
    "/content/logo_restec.png",
    "/content/mapas_restec/logo_restec.png"
)

'/content/mapas_restec/logo_restec.png'

In [189]:

# MALHA MUNICIPAL DO IBGE

def carregar_malha():

    arquivo_zip = PASTA_MALHA / "PR_Municipios_2022.zip"

    if not arquivo_zip.exists():
        resposta = requests.get(URL_MALHA, timeout=180)
        resposta.raise_for_status()
        arquivo_zip.write_bytes(resposta.content)

    with zipfile.ZipFile(arquivo_zip) as z:
        z.extractall(PASTA_MALHA)

    malha = gpd.read_file(
        next(PASTA_MALHA.rglob("*.shp"))
    )

    malha = malha.rename(
        columns={
            "NM_MUN": "municipio",
            "CD_MUN": "codigo_ibge"
        }
    )

    malha["chave_municipio"] = (
        malha["municipio"].map(padronizar_nome)
    )

    return malha


malha = carregar_malha()

print("Municípios na malha:", len(malha))


Municípios na malha: 399


In [190]:

# CRUZAMENTO E VALIDAÇÃO

mapa = malha.merge(
    residentes_mun,
    on="chave_municipio",
    how="left",
    validate="one_to_one"
)

mapa["residentes"] = mapa["residentes"].fillna(0)

mapa["possui_registro"] = mapa["residentes"] > 0

# Coordenadas dos municípios
pontos = mapa.geometry.representative_point()

mapa["x"] = pontos.x
mapa["y"] = pontos.y

# Validar correspondências
sem_correspondencia = (
    set(residentes_mun["chave_municipio"])
    - set(mapa["chave_municipio"])
)

assert not sem_correspondencia, (
    f"Municípios sem correspondência: {sem_correspondencia}"
)

print("VALIDAÇÃO ESPACIAL")
print("-------------------")
print("Total de municípios:", len(mapa))
print("Com registros:", int(mapa["possui_registro"].sum()))
print("Sem registros:", int((~mapa["possui_registro"]).sum()))
print("Total de residentes:", int(mapa["residentes"].sum()))

# Coordenadas geográficas
mapa_geo = mapa.to_crs("EPSG:4326")

# Dados para gráficos
dados_forca = (
    df.groupby("Forca", as_index=False)["residentes"]
    .sum()
    .sort_values("residentes", ascending=False)
)

dados_area = (
    df.groupby("area_atuacao", as_index=False)["residentes"]
    .sum()
    .sort_values("residentes", ascending=False)
)


VALIDAÇÃO ESPACIAL
-------------------
Total de municípios: 399
Com registros: 26
Sem registros: 373
Total de residentes: 335


In [191]:

# DASHBOARD INTERATIVO RESTEC

# Converter geometrias em GeoJSON
geojson_municipios = json.loads(
    mapa_geo.to_json()
)

# Dados municipais
dados_municipais = []

for _, linha in mapa_geo.iterrows():

    dados_municipais.append({
        "municipio": linha["municipio"],
        "residentes": int(linha["residentes"]),
        "possui_registro": bool(linha["possui_registro"])
    })

# Dados para filtros e gráficos
dados_completos = []

for _, linha in df.iterrows():

    dados_completos.append({
        "municipio": linha["MUN"],
        "forca": linha["Forca"],
        "area": linha["area_atuacao"],
        "residentes": int(linha["residentes"])
    })

# Indicadores
TOTAL_RESIDENTES = int(df["residentes"].sum())
TOTAL_MUNICIPIOS = len(mapa)
MUNICIPIOS_COM = int(mapa["possui_registro"].sum())
MUNICIPIOS_SEM = TOTAL_MUNICIPIOS - MUNICIPIOS_COM
TOTAL_FORCAS = df["Forca"].nunique()

# Serialização
GEOJSON = json.dumps(geojson_municipios)
DADOS_MUNICIPAIS = json.dumps(dados_municipais, ensure_ascii=False)
DADOS_COMPLETOS = json.dumps(dados_completos, ensure_ascii=False)

CHAVE = MAPTILER_KEY

In [192]:
# ============================================================
# INTERFACE DO DASHBOARD RESTEC–UNESPAR
# ============================================================

html = r"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">

<title>RESTEC | UNESPAR - Residentes</title>

<link rel="stylesheet"
href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>

<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>
* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: Arial, sans-serif;
    background: #F3F6F8;
    color: #263746;
}

/* CABEÇALHO */

.cabecalho {
    background: #6F9DAE;
    color: #FFFFFF;
    min-height: 120px;
    padding: 12px 32px;
    display: grid;
    grid-template-columns: 170px minmax(0, 1fr) 170px;
    align-items: center;
    gap: 22px;
    box-shadow: 0 2px 8px rgba(23,61,82,0.12);
}

.logo-esquerda,
.logo-direita {
    display: flex;
    align-items: center;
    justify-content: center;
    min-width: 0;
}

.logo-esquerda img,
.logo-direita img {
    display: block;
    max-width: 145px;
    max-height: 85px;
    width: auto;
    height: auto;
    object-fit: contain;
}

.titulo-cabecalho {
    text-align: center;
    min-width: 0;
}

.titulo-cabecalho h1 {
    margin: 0 0 8px;
    font-size: 18px;
    line-height: 1.35;
    font-weight: 600;
}

.titulo-cabecalho p {
    margin: 0 0 5px;
    font-size: 11px;
    letter-spacing: 1.3px;
    font-weight: 600;
}

.titulo-cabecalho span {
    font-size: 10px;
    letter-spacing: 1px;
}

/* CONTEÚDO */

.container {
    padding: 22px;
    max-width: 1600px;
    margin: auto;
}

/* FILTROS */

.filters {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 12px;
    margin-bottom: 18px;
}

select {
    padding: 11px 13px;
    border: 1px solid #D5DEE5;
    border-radius: 6px;
    background: white;
    width: 100%;
    color: #263746;
    font-size: 13px;
}

/* INDICADORES */

.cards {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 14px;
    margin-bottom: 18px;
}

.card {
    background: white;
    padding: 17px 19px;
    border-radius: 7px;
    border-left: 4px solid #477D91;
    box-shadow: 0 2px 7px #173D5210;
}

.card span {
    font-size: 11px;
    color: #71818C;
    letter-spacing: .3px;
}

.card h2 {
    margin: 9px 0 0;
    font-size: 27px;
    color: #173D52;
    font-weight: 600;
}

/* PAINÉIS */

.panel {
    background: white;
    padding: 22px;
    border-radius: 7px;
    margin-bottom: 18px;
    box-shadow: 0 2px 7px #173D5210;
    min-width: 0;
    overflow: hidden;
}

.panel h3 {
    margin: 0 0 15px;
    color: #173D52;
    font-size: 15px;
    font-weight: 600;
}

/* MAPA */

#map {
    width: 100%;
    height: 570px;
    border-radius: 5px;
    z-index: 1;
}

/* RÓTULOS DOS CÍRCULOS */

.rotulo-bola {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    color: #FFFFFF !important;
    font-size: 11px !important;
    font-weight: 800 !important;
    text-align: center;
    text-shadow:
        1px 1px 2px #173D52,
        -1px -1px 2px #173D52;
    padding: 0 !important;
    white-space: nowrap;
}

.rotulo-bola::before {
    display: none !important;
}

/* GRÁFICOS */

.charts {
    display: grid;
    grid-template-columns: minmax(0, 1fr) minmax(0, 1fr);
    gap: 18px;
    align-items: start;
}

.chart {
    width: 100%;
    min-width: 0;
    height: 390px;
}

/* GRÁFICO DE ÁREAS */

#graficoArea {
    height: 390px;
}

/* RODAPÉ */

footer {
    padding: 17px;
    text-align: center;
    color: #71818C;
    font-size: 10px;
    border-top: 1px solid #E1E7EB;
}

/* RESPONSIVIDADE */

@media(max-width: 1000px) {

    .cabecalho {
        grid-template-columns: 120px minmax(0, 1fr) 120px;
    }

    .logo-esquerda img,
    .logo-direita img {
        max-width: 110px;
        max-height: 75px;
    }

    .titulo-cabecalho h1 {
        font-size: 15px;
    }

    .charts {
        grid-template-columns: 1fr;
    }
}

@media(max-width: 850px) {

    .cards {
        grid-template-columns: repeat(2, 1fr);
    }

    .filters {
        grid-template-columns: 1fr;
    }

    #map {
        height: 450px;
    }
}

@media(max-width: 600px) {

    .cabecalho {
        grid-template-columns: 65px minmax(0, 1fr) 65px;
        padding: 10px;
        gap: 5px;
        min-height: 90px;
    }

    .logo-esquerda img,
    .logo-direita img {
        max-width: 60px;
        max-height: 45px;
    }

    .titulo-cabecalho h1 {
        font-size: 11px;
    }

    .titulo-cabecalho p {
        font-size: 8px;
        letter-spacing: .5px;
    }

    .titulo-cabecalho span {
        font-size: 7px;
    }

    .container {
        padding: 12px;
    }

    .cards {
        grid-template-columns: 1fr 1fr;
        gap: 9px;
    }

    .card {
        padding: 13px;
    }

    .card h2 {
        font-size: 22px;
    }

    .card span {
        font-size: 9px;
    }

    .panel {
        padding: 13px;
    }

    .chart {
        height: 430px;
    }

    #graficoArea {
        height: 390px;
    }
}
</style>
</head>

<body>

<!-- CABEÇALHO -->

<header class="cabecalho">

    <div class="logo-esquerda">
        <img src="logo_unespar.png" alt="UNESPAR">
    </div>

    <div class="titulo-cabecalho">

        <h1>
        Residência Técnica e Pós-Graduação em Gestão da Segurança Pública | RESTEC
        </h1>

        <p>DISTRIBUIÇÃO TERRITORIAL DE RESIDENTES</p>

        <span>UNESPAR · PARANÁ</span>

    </div>

    <div class="logo-direita">
        <img src="logo_restec.png" alt="RESTEC">
    </div>

</header>

<div class="container">

    <!-- FILTROS -->

    <div class="filters">

        <select id="filtroForca">
            <option value="">Todas as forças</option>
        </select>

        <select id="filtroArea">
            <option value="">Todas as áreas de atuação</option>
        </select>

        <select id="filtroMunicipio">
            <option value="">Todos os municípios</option>
        </select>

    </div>

    <!-- INDICADORES -->

    <div class="cards">

        <div class="card">
            <span>RESIDENTES</span>
            <h2 id="total">0</h2>
        </div>

        <div class="card">
            <span>MUNICÍPIOS COM REGISTROS</span>
            <h2 id="com">0</h2>
        </div>

        <div class="card">
            <span>FORÇAS REPRESENTADAS</span>
            <h2 id="forcas">0</h2>
        </div>

    </div>

    <!-- MAPA -->

    <div class="panel">

        <h3>Distribuição espacial dos residentes</h3>

        <div id="map"></div>

    </div>

    <!-- GRÁFICOS -->

    <div class="charts">

        <div class="panel">

            <h3>Residentes por força</h3>

            <div id="graficoForca" class="chart"></div>

        </div>

        <div class="panel">

            <h3>Residentes por área de atuação</h3>

            <div id="graficoArea" class="chart"></div>

        </div>

    </div>

    <div class="panel">

        <h3>Principais municípios</h3>

        <div id="graficoMunicipio" class="chart"></div>

    </div>

</div>

<footer>
    Fonte: Base RESTEC–UNESPAR; IBGE, Malha Municipal 2022.
    Mapa-base © MapTiler © OpenStreetMap contributors.
</footer>

<script>

// ============================================================
// DADOS
// ============================================================

const geojson = GEOJSON_PLACEHOLDER;
const dadosMunicipais = MUNICIPIOS_PLACEHOLDER;
const dados = DADOS_PLACEHOLDER;

// ============================================================
// MAPA
// ============================================================

const map = L.map('map').setView([-24.8, -51.5], 6);

L.tileLayer(
    'https://api.maptiler.com/maps/streets-v2/{z}/{x}/{y}.png?key=API_KEY_PLACEHOLDER',
    {
        attribution: '&copy; MapTiler &copy; OpenStreetMap contributors',
        maxZoom: 18
    }
).addTo(map);

let camadaPoligonos;
let camadaPontos;

const cores = {
    azul: '#477D91',
    cinza: '#C8D0D5'
};

function numero(v) {
    return Number(v).toLocaleString('pt-BR');
}

// ============================================================
// PREENCHIMENTO DOS FILTROS
// ============================================================

function preencher(id, valores) {

    const el = document.getElementById(id);

    [...new Set(valores.filter(v => v))].sort().forEach(v => {

        const op = document.createElement('option');

        op.value = v;
        op.textContent = v;

        el.appendChild(op);

    });
}

preencher('filtroForca', dados.map(d => d.forca));
preencher('filtroArea', dados.map(d => d.area));
preencher('filtroMunicipio', dados.map(d => d.municipio));

// ============================================================
// FILTRAGEM
// ============================================================

function filtrar() {

    const forca = document.getElementById('filtroForca').value;
    const area = document.getElementById('filtroArea').value;
    const municipio = document.getElementById('filtroMunicipio').value;

    return dados.filter(d =>
        (!forca || d.forca === forca) &&
        (!area || d.area === area) &&
        (!municipio || d.municipio === municipio)
    );
}

// ============================================================
// GRÁFICOS
// ============================================================

function grafico(
    id,
    objeto,
    cor,
    alturaMinima = 390,
    limite = null
) {

    let entradas = Object.entries(objeto)
        .filter(([nome, valor]) => nome && valor > 0)
        .sort((a, b) => b[1] - a[1]);

    // LIMITAR QUANTIDADE DE CATEGORIAS
    if (limite !== null) {
        entradas = entradas.slice(0, limite);
    }

    const categorias = entradas.map(x => x[0]);
    const valores = entradas.map(x => x[1]);

    // ALTURA FIXA PARA O GRÁFICO DE ÁREAS
    const altura = limite === 10
        ? 390
        : Math.max(alturaMinima, categorias.length * 48 + 100);

    const elemento = document.getElementById(id);
    elemento.style.height = altura + 'px';

    const maiorValor = Math.max(...valores, 1);

    const coresBarras = valores.map(valor => {
        const intensidade = valor / maiorValor;

        return intensidade > 0.65 ? cor :
               intensidade > 0.30 ? '#54899A' :
               '#A9C3CC';
    });

    Plotly.react(id, [{

        type: 'bar',
        orientation: 'h',

        x: valores,
        y: categorias,

        marker: {
            color: coresBarras,
            line: {
                width: 0
            }
        },

        text: valores.map(valor => numero(valor)),

        textposition: 'outside',

        textfont: {
            size: 11,
            color: '#263746',
            family: 'Arial'
        },

        cliponaxis: false,

        hovertemplate:
            '<b>%{y}</b><br>' +
            'Residentes: %{x:,}' +
            '<extra></extra>'

    }], {

        height: altura,
        autosize: true,

        margin: {
            l: id === 'graficoArea' ? 175 : 150,
            r: 55,
            t: 15,
            b: 45
        },

        paper_bgcolor: '#FFFFFF',
        plot_bgcolor: '#FFFFFF',

        bargap: limite === 10 ? 0.30 : 0.38,

        xaxis: {

            title: {
                text: 'Número de residentes',
                font: {
                    size: 10,
                    color: '#71818C'
                },
                standoff: 10
            },

            showgrid: true,
            gridcolor: '#E9EFF2',
            zeroline: false,
            showline: false,

            tickfont: {
                size: 10,
                color: '#71818C'
            },

            automargin: true,
            rangemode: 'tozero'

        },

        yaxis: {

            autorange: 'reversed',
            showgrid: false,
            showline: false,
            ticks: '',
            automargin: true,

            tickfont: {
                family: 'Arial',
                size: 11,
                color: '#34495E'
            }

        },

        font: {
            family: 'Arial, sans-serif',
            color: '#263746'
        },

        showlegend: false,

        hoverlabel: {
            bgcolor: '#173D52',
            font: {
                color: '#FFFFFF',
                size: 12
            }
        }

    }, {

        responsive: true,
        displayModeBar: false,
        staticPlot: false

    });

}

// ============================================================
// ATUALIZAÇÃO DO DASHBOARD
// ============================================================

function atualizar() {

    const filtrados = filtrar();

    const agrupado = {};
    const forcas = {};
    const areas = {};

    filtrados.forEach(d => {

        const residentes = Number(d.residentes) || 0;

        agrupado[d.municipio] =
            (agrupado[d.municipio] || 0) + residentes;

        forcas[d.forca] =
            (forcas[d.forca] || 0) + residentes;

        areas[d.area] =
            (areas[d.area] || 0) + residentes;

    });

    const total = filtrados.reduce(
        (s, d) => s + (Number(d.residentes) || 0), 0
    );

    // INDICADORES

    const municipiosAtivos = dadosMunicipais.filter(m =>
        (agrupado[m.municipio] || 0) > 0
    ).length;

    document.getElementById('total').textContent =
        numero(total);

    document.getElementById('com').textContent =
        numero(municipiosAtivos);

    document.getElementById('forcas').textContent =
        numero(Object.keys(forcas).filter(k => forcas[k] > 0).length);

    // POLÍGONOS MUNICIPAIS

    if (camadaPoligonos) map.removeLayer(camadaPoligonos);
    if (camadaPontos) map.removeLayer(camadaPontos);

    camadaPoligonos = L.geoJSON(geojson, {

        style: feature => {

            const nome = feature.properties.municipio;
            const valor = agrupado[nome] || 0;

            return {

                fillColor: valor > 0 ? cores.azul : cores.cinza,

                weight: 0.7,

                color: '#FFFFFF',

                fillOpacity: valor > 0 ? 0.42 : 0.35

            };
        },

        onEachFeature: (feature, layer) => {

            const nome = feature.properties.municipio;
            const valor = agrupado[nome] || 0;

            layer.bindPopup(

                '<b>' + nome + '</b><br>' +
                'Residentes: ' + numero(valor) + '<br>' +
                (valor > 0 ? 'Com registros' : 'Sem registros')

            );
        }

    }).addTo(map);

    // SÍMBOLOS PROPORCIONAIS

    camadaPontos = L.layerGroup();

    dadosMunicipais.forEach(m => {

        const valor = agrupado[m.municipio] || 0;

        if (valor <= 0) return;

        const feature = geojson.features.find(
            f => f.properties.municipio === m.municipio
        );

        if (!feature) return;

        const camadaTemporaria = L.geoJSON(feature);
        const centro = camadaTemporaria.getBounds().getCenter();

        const raio = Math.max(
            13,
            Math.min(29, Math.sqrt(valor) * 2.2)
        );

        const circulo = L.circleMarker(centro, {

            radius: raio,

            color: '#FFFFFF',

            weight: 1.5,

            fillColor: '#14566B',

            fillOpacity: 0.92

        });

        circulo.bindTooltip(
            numero(valor),
            {
                permanent: true,
                direction: 'center',
                className: 'rotulo-bola',
                opacity: 1
            }
        );

        circulo.bindPopup(
            '<b>' + m.municipio + '</b><br>' +
            'Residentes: ' + numero(valor)
        );

        circulo.addTo(camadaPontos);

    });

    camadaPontos.addTo(map);

    // GRÁFICO POR FORÇA

    grafico(
        'graficoForca',
        forcas,
        '#477D91',
        390
    );

    // GRÁFICO POR ÁREA DE ATUAÇÃO
    // SOMENTE AS 10 ÁREAS COM MAIS RESIDENTES

    grafico(
        'graficoArea',
        areas,
        '#2A7188',
        390,
        10
    );

    // PRINCIPAIS MUNICÍPIOS

    const principais = Object.entries(agrupado)
        .filter(x => x[1] > 0)
        .sort((a, b) => b[1] - a[1])
        .slice(0, 10);

    grafico(
        'graficoMunicipio',
        Object.fromEntries(principais),
        '#3977A8',
        480
    );

}

// ============================================================
// EVENTOS DOS FILTROS
// ============================================================

['filtroForca','filtroArea','filtroMunicipio'].forEach(id => {

    document.getElementById(id).addEventListener(
        'change',
        atualizar
    );

});

// ============================================================
// EXECUÇÃO INICIAL
// ============================================================

atualizar();

</script>

</body>
</html>
"""

# ============================================================
# SUBSTITUIÇÃO DOS DADOS
# ============================================================

html = (
    html
    .replace("GEOJSON_PLACEHOLDER", GEOJSON)
    .replace("MUNICIPIOS_PLACEHOLDER", DADOS_MUNICIPAIS)
    .replace("DADOS_PLACEHOLDER", DADOS_COMPLETOS)
    .replace("API_KEY_PLACEHOLDER", CHAVE)
)

# ============================================================
# SALVAR DASHBOARD
# ============================================================

ARQUIVO_DASHBOARD = PASTA_SAIDA / "index.html"

ARQUIVO_DASHBOARD.write_text(
    html,
    encoding="utf-8"
)

print("Dashboard criado com sucesso!")
print("Arquivo:", ARQUIVO_DASHBOARD)

Dashboard criado com sucesso!
Arquivo: /content/mapas_restec/index.html


In [193]:

# VISUALIZAÇÃO E EXPORTAÇÃO

from IPython.display import IFrame, display
from google.colab import files

display(
    IFrame(
        src=str(ARQUIVO_DASHBOARD),
        width="100%",
        height=850
    )
)

files.download(str(ARQUIVO_DASHBOARD))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>